In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
file_path = "olmo_generation_output_RAG.csv"
df = pd.read_csv(file_path, encoding='ISO-8859-1's)
df.head()


,question,true_answer,OLMo-1B,RAG_OLMo-1B,OLMo-1B-0724-hf,RAG_OLMo-1B-0724-hf,OLMo-2-0425-1B-Instruct,RAG_OLMo-2-0425-1B-Instruct
0,"Hi, \nIâm following this tutorial: The LSST...",Quick comment on the code: \n \n \n \n petarz...,"Hi, \nIâm following this tutorial: The LSST...",You are an astrophysics expert. Please answer ...,"Hi, \nIâm following this tutorial: The LSST...",You are an astrophysics expert. Please answer ...,"Hi, \nIâm following this tutorial: The LSST...",You are an astrophysics expert. Please answer ...
1,I have the following C++ class : \n class CcdI...,After several iteration with @ktl and @rowe...,I have the following C++ class : \n class CcdI...,You are an astrophysics expert. Please answer ...,I have the following C++ class : \n class CcdI...,You are an astrophysics expert. Please answer ...,I have the following C++ class : \n class CcdI...,You are an astrophysics expert. Please answer ...
2,Question on how forced photometry will be run ...,I take this to mean that a DIASource which is ...,Question on how forced photometry will be run ...,You are an astrophysics expert. Please answer ...,Question on how forced photometry will be run ...,You are an astrophysics expert. Please answer ...,Question on how forced photometry will be run ...,You are an astrophysics expert. Please answer ...
3,"Hi there, \n Is there some way I find out what...",Hi James \nmaybe \n dafButler.Butler.get_known...,"Hi there, \n Is there some way I find out what...",You are an astrophysics expert. Please answer ...,"Hi there, \n Is there some way I find out what...",You are an astrophysics expert. Please answer ...,"Hi there, \n Is there some way I find out what...",You are an astrophysics expert. Please answer ...
4,Iâm having trouble building FFTW with texinf...,This has now been fixed and 3.3.4 is the curre...,Iâm having trouble building FFTW with texinf...,You are an astrophysics expert. Please answer ...,Iâm having trouble building FFTW with texinf...,You are an astrophysics expert. Please answer ...,Iâm having trouble building FFTW with texinf...,You are an astrophysics expert. Please answer ...


In [6]:
# generated answer columns
generated_columns = [
    'OLMo-1B',
    'RAG_OLMo-1B',
    'OLMo-1B-0724-hf',
    'RAG_OLMo-1B-0724-hf',
    'OLMo-2-0425-1B-Instruct',
    'RAG_OLMo-2-0425-1B-Instruct'
]

# cosine similarities for each model vs true_answer
cosine_scores = {}
true_answers = df['true_answer'].fillna("")

for col in generated_columns:
    generated_answers = df[col].fillna("")
    combined = list(true_answers) + list(generated_answers)

    # Vectorize text
    tfidf = TfidfVectorizer().fit_transform(combined)
    true_vecs = tfidf[:len(df)]
    gen_vecs = tfidf[len(df):]

    # Calculate cosine similarities row-wise
    scores = cosine_similarity(true_vecs, gen_vecs).diagonal()
    cosine_scores[col] = scores




In [9]:
cosine_scores


{'OLMo-1B': array([0.21242606, 0.15862734, 0.61173689, 0.19472723, 0.09103294]),
 'RAG_OLMo-1B': array([0.30451033, 0.19376255, 0.6173611 , 0.06663405, 0.12641022]),
 'OLMo-1B-0724-hf': array([0.23805775, 0.12230109, 0.68145807, 0.15069681, 0.0985959 ]),
 'RAG_OLMo-1B-0724-hf': array([0.29742343, 0.23284278, 0.60016154, 0.0860829 , 0.14538606]),
 'OLMo-2-0425-1B-Instruct': array([0.20068078, 0.16508658, 0.68923934, 0.22468449, 0.10253033]),
 'RAG_OLMo-2-0425-1B-Instruct': array([0.31176303, 0.21157845, 0.68377712, 0.08791098, 0.11867541])}

In [ ]:
# Add cosine similarity columns to the dataframe
for model_name, scores in cosine_scores.items():
    cosine_col = f"{model_name}_cosine"
    df[cosine_col] = scores

df.to_csv("olmo_generation_output_with_cosine.csv", index=False)

df.head()

,question,true_answer,OLMo-1B,RAG_OLMo-1B,OLMo-1B-0724-hf,RAG_OLMo-1B-0724-hf,OLMo-2-0425-1B-Instruct,RAG_OLMo-2-0425-1B-Instruct,OLMo-1B_cosine,RAG_OLMo-1B_cosine,OLMo-1B-0724-hf_cosine,RAG_OLMo-1B-0724-hf_cosine,OLMo-2-0425-1B-Instruct_cosine,RAG_OLMo-2-0425-1B-Instruct_cosine
0,"Hi, \nIâm following this tutorial: The LSST...",Quick comment on the code: \n \n \n \n petarz...,"Hi, \nIâm following this tutorial: The LSST...",You are an astrophysics expert. Please answer ...,"Hi, \nIâm following this tutorial: The LSST...",You are an astrophysics expert. Please answer ...,"Hi, \nIâm following this tutorial: The LSST...",You are an astrophysics expert. Please answer ...,0.212426,0.304510,0.238058,0.297423,0.200681,0.311763
1,I have the following C++ class : \n class CcdI...,After several iteration with @ktl and @rowe...,I have the following C++ class : \n class CcdI...,You are an astrophysics expert. Please answer ...,I have the following C++ class : \n class CcdI...,You are an astrophysics expert. Please answer ...,I have the following C++ class : \n class CcdI...,You are an astrophysics expert. Please answer ...,0.158627,0.193763,0.122301,0.232843,0.165087,0.211578
2,Question on how forced photometry will be run ...,I take this to mean that a DIASource which is ...,Question on how forced photometry will be run ...,You are an astrophysics expert. Please answer ...,Question on how forced photometry will be run ...,You are an astrophysics expert. Please answer ...,Question on how forced photometry will be run ...,You are an astrophysics expert. Please answer ...,0.611737,0.617361,0.681458,0.600162,0.689239,0.683777
3,"Hi there, \n Is there some way I find out what...",Hi James \nmaybe \n dafButler.Butler.get_known...,"Hi there, \n Is there some way I find out what...",You are an astrophysics expert. Please answer ...,"Hi there, \n Is there some way I find out what...",You are an astrophysics expert. Please answer ...,"Hi there, \n Is there some way I find out what...",You are an astrophysics expert. Please answer ...,0.194727,0.066634,0.150697,0.086083,0.224684,0.087911
4,Iâm having trouble building FFTW with texinf...,This has now been fixed and 3.3.4 is the curre...,Iâm having trouble building FFTW with texinf...,You are an astrophysics expert. Please answer ...,Iâm having trouble building FFTW with texinf...,You are an astrophysics expert. Please answer ...,Iâm having trouble building FFTW with texinf...,You are an astrophysics expert. Please answer ...,0.091033,0.126410,0.098596,0.145386,0.102530,0.118675


In [12]:
# Creating a new DataFrame to show average similarity per model
cosine_summary = pd.DataFrame({
    "Model": list(cosine_scores.keys()),
    "Average Cosine Similarity": [scores.mean() for scores in cosine_scores.values()]
})
cosine_summary

,Model,Average Cosine Similarity
0,OLMo-1B,0.253710
1,RAG_OLMo-1B,0.261736
2,OLMo-1B-0724-hf,0.258222
3,RAG_OLMo-1B-0724-hf,0.272379
4,OLMo-2-0425-1B-Instruct,0.276444
5,RAG_OLMo-2-0425-1B-Instruct,0.282741
